# Beyond Words — Notebook 07: EmoSen split-shift diagnostic (CPU, minutes)

**Why this exists.** Condition B (XLM-R + EmoSen) scored about the same on EmoSen test as Condition A, which never saw EmoSen. A model trained on in-domain data normally gains a lot there. B's predictions show why this is not a clean "no effect": it over-predicts *disgust* (about 36% of EmoSen-test predictions, against 5% gold) and most gold *anger* tweets are predicted as disgust.

In the original EmoSen splits, the label priors differ sharply between train, validation and test (for example disgust is 7.5% of train rows but 1.8% of validation rows; this is in the unmodified files, not a result of our deduplication). This notebook measures, with a cheap linear model:

1. **Prior shift:** label shares per split, and the total-variation distance between split priors.
2. **Learnability and cross-split consistency:** a TF-IDF + logistic-regression classifier is trained on one split and scored on every split (5-fold cross-validation when the same split is used for both). If within-train cross-validation is much higher than train→validation / train→test, while validation→test is high, the train split's labels or content differ from the evaluation splits.

It trains nothing in B or D, touches no Condition A–D files, and uses only the frozen EmoSen CSV. Scoring uses the four classes with enough support in every split: anger, disgust, joy, sadness. EmoSen text stays in this runtime and is not saved.

**How to use the result.** This is analysis of why adaptation does not transfer. It is not a basis for changing the B/D recipe after seeing test results. Any recipe change (re-weighting, resampling, relabelling) must be run for both backbones as a separately labelled ablation.


## Section 1 — Setup and load

Packages: `scikit-learn`, `pandas`, `numpy` (all preinstalled in Colab). The data-loader below holds the `EMOSEN_PERMISSION_CONFIRMED` gate and validates the CSV against the frozen dedup audit.

In [7]:
!mkdir -p /content/Beyond-Words/data/processed
!cp -r /content/drive/MyDrive/Beyond-Words/data/processed/. /content/Beyond-Words/data/processed/

In [6]:
!cp -r /content/drive/MyDrive/Beyond-Words/data/processed /content/Beyond-Words/data/

cp: cannot create directory '/content/Beyond-Words/data/': No such file or directory


In [8]:
import numpy as np, pandas as pd, json, sys
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline, make_union
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import f1_score, accuracy_score

try:
    import google.colab  # noqa: F401
    PROJECT_ROOT = Path("/content/Beyond-Words")
except ImportError:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EMOSEN_PATH = PROJECT_ROOT / "data" / "processed" / "emosen_dedup_frozen.csv"
LABEL_ORDER = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]
ID2LABEL = dict(enumerate(LABEL_ORDER)); LABEL2ID = {l: i for i, l in enumerate(LABEL_ORDER)}
NUM_LABELS = 7

# ── EmoSen (frozen, deduplicated) and optional HinEmo-9 gold loaders ─────────
EMOSEN_PERMISSION_CONFIRMED = True  # set True ONLY with written confirmation covering this processing

SIX_NAMES = LABEL_ORDER[:6]
EXPECTED_EMOSEN_SIX = {   # retained rows per split after dedup, Others excluded (from the frozen audit)
    "train"     : {"anger": 2037, "disgust": 1016, "fear": 56, "joy": 3712, "sadness": 823, "surprise": 50},
    "validation": {"anger": 601,  "disgust": 55,   "fear": 4,  "joy": 961,  "sadness": 299, "surprise": 6},
    "test"      : {"anger": 677,  "disgust": 105,  "fear": 13, "joy": 1004, "sadness": 121, "surprise": 7},
}
EXPECTED_EMOSEN_OTHERS = {"train": 5828, "validation": 1036, "test": 1060}

def load_emosen_six(path):
    """Load the frozen EmoSen CSV, validate it against the frozen audit, drop Others, return six-class rows."""
    raw = pd.read_csv(path, dtype=str, keep_default_na=False)
    pick = lambda opts: next((c for c in opts if c in raw.columns), None)
    c_split, c_id = pick(["split"]), pick(["id", "example_id"])
    c_text, c_lab = pick(["tweet_fixed", "text"]), pick(["emotion", "mapped_label", "label"])
    if None in (c_split, c_id, c_text, c_lab):
        raise ValueError(f"EmoSen CSV needs split/id/tweet_fixed/emotion columns; found {list(raw.columns)}")
    d = pd.DataFrame({
        "split"  : raw[c_split].str.strip().str.lower().replace({"val": "validation", "dev": "validation"}),
        "id"     : raw[c_id].str.strip(),
        "text"   : raw[c_text],
        "emotion": raw[c_lab].str.strip().str.lower(),
    })
    if set(d["split"]) != {"train", "validation", "test"}:
        raise ValueError(f"Unexpected EmoSen splits: {sorted(set(d['split']))}")
    if d.duplicated(["split", "id"]).any():
        raise ValueError("Duplicate (split, id) rows in the EmoSen CSV.")
    unknown = set(d["emotion"]) - set(SIX_NAMES) - {"others"}
    if unknown:
        raise ValueError(f"Unexpected EmoSen emotion labels: {sorted(unknown)}")
    if (d["text"].str.strip() == "").any():
        raise ValueError("Empty EmoSen text found.")

    others = d[d["emotion"] == "others"].groupby("split").size().to_dict()
    six    = d[d["emotion"] != "others"].copy()
    got    = {s: six[six["split"] == s].groupby("emotion").size().to_dict() for s in EXPECTED_EMOSEN_SIX}
    problems = []
    for s, exp in EXPECTED_EMOSEN_SIX.items():
        if got[s] != exp:
            problems.append(f"{s}: got {got[s]}, expected {exp}")
        if others.get(s, 0) != EXPECTED_EMOSEN_OTHERS[s]:
            problems.append(f"{s} Others: got {others.get(s, 0)}, expected {EXPECTED_EMOSEN_OTHERS[s]}")
    if problems:
        raise RuntimeError("EmoSen CSV does not match the frozen dedup audit:\n  " + "\n  ".join(problems))

    six["label_id"] = six["emotion"].map(LABEL2ID)
    return six.reset_index(drop=True), others

EXPECTED_HINEMO_RAW     = 393
EXPECTED_HINEMO_KEPT    = {"anger": 35, "fear": 49, "joy": 38, "sadness": 26, "surprise": 122}
EXPECTED_HINEMO_DROPPED = {"love": 41, "devotion": 31, "neutral": 26, "nostalgia": 25}

def load_hinemo_optional(path, strict=True):
    """Optional prepared HinEmo-9 gold Hinglish test file: columns id,text,label.
    Validates the raw file (393 rows, unique ids, non-empty text, expected label counts) before filtering to the
    five shared classes, so an accidentally replaced CSV cannot silently change the test set."""
    if not Path(path).exists():
        return None, {}
    raw = pd.read_csv(path, dtype=str, keep_default_na=False)
    for c in ("id", "text", "label"):
        if c not in raw.columns:
            raise ValueError(f"HinEmo file needs columns id,text,label; found {list(raw.columns)}")
    lab = raw["label"].str.strip().str.lower()
    if strict:
        problems = []
        if len(raw) != EXPECTED_HINEMO_RAW:
            problems.append(f"rows: got {len(raw)}, expected {EXPECTED_HINEMO_RAW}")
        if raw["id"].str.strip().duplicated().any():
            problems.append("duplicate ids")
        if (raw["text"].str.strip() == "").any():
            problems.append("empty text")
        got = lab.value_counts().to_dict()
        want = {**EXPECTED_HINEMO_KEPT, **EXPECTED_HINEMO_DROPPED}
        if got != want:
            problems.append(f"label counts: got {got}, expected {want}")
        if problems:
            raise RuntimeError("HinEmo-9 CSV does not match the audited gold test split:\n  " + "\n  ".join(problems))
    dropped = lab[~lab.isin(SIX_NAMES)].value_counts().to_dict()
    d = raw[lab.isin(SIX_NAMES)].copy()
    d["label_id"] = lab[lab.isin(SIX_NAMES)].map(LABEL2ID)
    d = d.rename(columns={"id": "example_id"})[["example_id", "text", "label_id"]].reset_index(drop=True)
    return d, dropped


if not EMOSEN_PERMISSION_CONFIRMED:
    raise PermissionError("Set EMOSEN_PERMISSION_CONFIRMED = True in this cell only with written confirmation.")
hi_six, hi_others = load_emosen_six(EMOSEN_PATH)
print("Loaded EmoSen six-class rows:", len(hi_six), "| Others rows:", hi_others)


Loaded EmoSen six-class rows: 11547 | Others rows: {'test': 1060, 'train': 5828, 'validation': 1036}


In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Section 2 — Label priors per split

In [10]:
SPLITS = ["train", "validation", "test"]
tab = pd.crosstab(hi_six["emotion"], hi_six["split"]).reindex(index=SIX_NAMES, columns=SPLITS)
for s in SPLITS:
    tab.loc["others", s] = hi_others[s]
shares = tab / tab.sum()
print("Counts (this file: deduplicated, Others included for context):"); print(tab.astype(int))
print("\nShares of all rows:"); print(shares.round(3))
six_sh = tab.loc[SIX_NAMES] / tab.loc[SIX_NAMES].sum()
print("\nShares among the six classes only:"); print(six_sh.round(3))
tv = pd.DataFrame(index=SPLITS, columns=SPLITS, dtype=float)
for a in SPLITS:
    for b in SPLITS:
        tv.loc[a, b] = 0.5 * np.abs(six_sh[a] - six_sh[b]).sum()
print("\nTotal-variation distance between six-class label priors (0 = identical, 1 = disjoint):")
print(tv.round(3))


Counts (this file: deduplicated, Others included for context):
split     train  validation  test
emotion                          
anger      2037         601   677
disgust    1016          55   105
fear         56           4    13
joy        3712         961  1004
sadness     823         299   121
surprise     50           6     7
others     5828        1036  1060

Shares of all rows:
split     train  validation   test
emotion                           
anger     0.151       0.203  0.227
disgust   0.075       0.019  0.035
fear      0.004       0.001  0.004
joy       0.275       0.324  0.336
sadness   0.061       0.101  0.041
surprise  0.004       0.002  0.002
others    0.431       0.350  0.355

Shares among the six classes only:
split     train  validation   test
emotion                           
anger     0.265       0.312  0.351
disgust   0.132       0.029  0.054
fear      0.007       0.002  0.007
joy       0.482       0.499  0.521
sadness   0.107       0.155  0.063
surprise  0.00

## Section 3 — Cross-split learnability (TF-IDF + logistic regression)

In [11]:
CORE4 = ["anger", "disgust", "joy", "sadness"]
core = hi_six[hi_six["emotion"].isin(CORE4)].copy()
parts = {s: core[core["split"] == s].reset_index(drop=True) for s in SPLITS}

def make_model():
    feats = make_union(
        TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True, max_features=200000),
        TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, sublinear_tf=True, lowercase=True),
    )
    return make_pipeline(feats, LogisticRegression(C=3.0, max_iter=2000, class_weight="balanced"))

def score(y_true, y_pred):
    return {"macro_f1": f1_score(y_true, y_pred, labels=CORE4, average="macro", zero_division=0),
            "accuracy": accuracy_score(y_true, y_pred),
            **{f"f1_{c}": f1_score(y_true, y_pred, labels=[c], average="macro", zero_division=0) for c in CORE4}}

res, per_class, pred_dist = {}, {}, {}
for tr in SPLITS:
    for ev in SPLITS:
        if tr == ev:
            skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
            y_pred = cross_val_predict(make_model(), parts[tr]["text"], parts[tr]["emotion"], cv=skf)
        else:
            m = make_model().fit(parts[tr]["text"], parts[tr]["emotion"])
            y_pred = m.predict(parts[ev]["text"])
        sc = score(parts[ev]["emotion"], y_pred)
        res[(tr, ev)] = sc["macro_f1"]; per_class[(tr, ev)] = sc
        pred_dist[(tr, ev)] = pd.Series(y_pred).value_counts(normalize=True).reindex(CORE4).fillna(0)

mat = pd.DataFrame([[res[(t, e)] for e in SPLITS] for t in SPLITS], index=[f"train on {s}" for s in SPLITS],
                   columns=[f"score on {s}" for s in SPLITS])
print("Four-class macro-F1 (anger, disgust, joy, sadness). Diagonal = 5-fold cross-validation within the split:")
print(mat.round(3))
print("\nMajority-class accuracy per split (always predict the most frequent core class):")
print({s: round(parts[s]["emotion"].value_counts(normalize=True).iloc[0], 3) for s in SPLITS})
print("\nPer-class F1, train on train -> score on test, versus 5-fold cross-validation within train:")
print(pd.DataFrame({"train->test": {c: per_class[("train", "test")][f"f1_{c}"] for c in CORE4},
                    "CV within train": {c: per_class[("train", "train")][f"f1_{c}"] for c in CORE4},
                    "validation->test": {c: per_class[("validation", "test")][f"f1_{c}"] for c in CORE4}}).round(3))
print("\nPredicted label shares, train on train -> score on test (gold shares on test shown last):")
print(pd.DataFrame({"predicted": pred_dist[("train", "test")],
                    "gold test": parts["test"]["emotion"].value_counts(normalize=True).reindex(CORE4)}).round(3))

print("\nHow to read this:")
print(" - Diagonal much higher than the train row's off-diagonal cells: the train split's labels/content differ from the other splits.")
print(" - 'validation on test' close to the validation diagonal: validation and test are consistent with each other.")
print(" - A large disgust share in 'predicted' versus 'gold test': the train split's disgust covers content the test split labels differently (for example anger).")
mat.to_csv(PROJECT_ROOT / "emosen_split_shift_matrix.csv")


Four-class macro-F1 (anger, disgust, joy, sadness). Diagonal = 5-fold cross-validation within the split:
                     score on train  score on validation  score on test
train on train                0.520                0.500          0.528
train on validation           0.467                0.551          0.564
train on test                 0.447                0.556          0.564

Majority-class accuracy per split (always predict the most frequent core class):
{'train': np.float64(0.489), 'validation': np.float64(0.502), 'test': np.float64(0.526)}

Per-class F1, train on train -> score on test, versus 5-fold cross-validation within train:
         train->test  CV within train  validation->test
anger          0.662            0.561             0.748
disgust        0.195            0.290             0.283
joy            0.924            0.909             0.913
sadness        0.330            0.321             0.310

Predicted label shares, train on train -> score on test (gold 